# Algorithmic Trading and Quantitative Strategies
## Part 6: Backtrader — Money Management, Optimization, and Walk-Forward Analysis
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. [Money Management in Backtrader](#1.-Money-Management-in-Backtrader)
2. [Parameter Optimization](#2.-Parameter-Optimization)
3. [Walk-Forward Analysis](#3.-Walk-Forward-Analysis)
4. [Biases in Strategy Development](#4.-Biases-in-Strategy-Development)
5. [Strategy Design Methodology](#5.-Strategy-Design-Methodology)
6. [Exercises](#6.-Exercises)

## 1. Money Management in Backtrader

Money management (or risk management) determines *how much* to trade, while the strategy determines *when* to trade. Proper money management can be the difference between a profitable and a losing strategy.

### 1.1 Stop Loss and Take Profit

A **stop loss** limits downside risk by automatically closing a position when the price moves against you by a specified amount. A **take profit** locks in gains when price reaches a target.

In backtrader, these are implemented using **bracket orders** — a main entry order packaged with a stop loss and take profit order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import backtrader as bt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Download data
price_data = yf.download('TUR', start='2008-01-01', end='2026-12-31')
price_data.columns = price_data.columns.droplevel('Ticker')
print(f"Downloaded {len(price_data)} bars for SPY")

In [ ]:
class BracketOrderStrategy(bt.Strategy):
    """
    SMA Crossover with bracket orders (stop loss + take profit).
    """
    params = dict(
        fast_period=20,
        slow_period=50,
        stop_loss_pct=0.03,     # 3% stop loss
        take_profit_pct=0.06,   # 6% take profit
        printlog=True,
    )
    
    def __init__(self):
        self.sma_fast = bt.ind.SMA(self.data.close, period=self.p.fast_period)
        self.sma_slow = bt.ind.SMA(self.data.close, period=self.p.slow_period)
        self.crossover = bt.ind.CrossOver(self.sma_fast, self.sma_slow)
        self.order = None
        self.trade_count = 0
    
    def log(self, txt):
        if self.p.printlog:
            print(f"{self.data.datetime.date(0)}: {txt}")
    
    def notify_order(self, order):
        if order.status == order.Completed:
            if order.isbuy():
                self.log(f"BUY @ {order.executed.price:.2f}")
            else:
                self.log(f"SELL @ {order.executed.price:.2f}")
        self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            self.trade_count += 1
            self.log(f"TRADE #{self.trade_count}: PnL={trade.pnlcomm:.2f}")
    
    def next(self):
        if self.order or self.position:
            return
        
        if self.crossover > 0:
            price = self.data.close[0]
            stop_price = price * (1 - self.p.stop_loss_pct)
            take_profit_price = price * (1 + self.p.take_profit_pct)
            
            self.order = self.buy_bracket(
                limitprice=take_profit_price,
                stopprice=stop_price,
                size=int(self.broker.getcash() * 0.95 / price),
            )
            self.log(f"BRACKET ORDER: Entry~{price:.2f}, "
                     f"SL={stop_price:.2f}, TP={take_profit_price:.2f}")

# Run
cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=price_data))
cerebro.addstrategy(BracketOrderStrategy, printlog=False)
cerebro.broker.setcash(100000)
cerebro.broker.setcommission(commission=0.001)

cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='dd')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')

print(f"Starting: ${cerebro.broker.getvalue():,.2f}")
results = cerebro.run()
strat = results[0]
print(f"Final:    ${cerebro.broker.getvalue():,.2f}")

# Print analysis
ret = strat.analyzers.returns.get_analysis()
dd = strat.analyzers.dd.get_analysis()
ta = strat.analyzers.trades.get_analysis()
print(f"\nTotal Return: {ret.get('rtot',0):.2%}")
print(f"Max Drawdown: {dd.max.drawdown:.2f}%")
print(f"Total Trades: {ta.total.total}")

### 1.2 Trailing Stop Orders

A **trailing stop** adjusts the stop price as the position moves in your favor, locking in profits while allowing the trade to continue.

In backtrader, use `StopTrail` with either a fixed dollar amount or a percentage:
- `self.sell(exectype=bt.Order.StopTrail, trailpercent=0.05)` — 5% trailing stop
- `self.sell(exectype=bt.Order.StopTrail, trailamount=5.0)` — $5 trailing stop

In [ ]:
class TrailingStopStrategy(bt.Strategy):
    """SMA crossover entry with trailing stop exit."""
    params = dict(
        fast_period=20,
        slow_period=50,
        trail_pct=0.05,
    )
    
    def __init__(self):
        self.sma_fast = bt.ind.SMA(self.data.close, period=self.p.fast_period)
        self.sma_slow = bt.ind.SMA(self.data.close, period=self.p.slow_period)
        self.crossover = bt.ind.CrossOver(self.sma_fast, self.sma_slow)
        self.order = None
        self.stop_order = None
        self.trade_count = 0
    
    def notify_order(self, order):
        if order.status == order.Completed:
            if order.isbuy():
                self.stop_order = self.sell(
                    exectype=bt.Order.StopTrail,
                    trailpercent=self.p.trail_pct
                )
            elif order.issell() and self.stop_order:
                self.stop_order = None
        if order.status in [order.Completed, order.Canceled, order.Rejected]:
            self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            self.trade_count += 1
            self.stop_order = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.crossover > 0:
                size = int(self.broker.getcash() * 0.95 / self.data.close[0])
                self.order = self.buy(size=size)
        else:
            if self.crossover < 0:
                if self.stop_order:
                    self.cancel(self.stop_order)
                    self.stop_order = None
                self.order = self.close()
    
    def stop(self):
        print(f"Trailing Stop ({self.p.trail_pct:.0%}): "
              f"{self.trade_count} trades, "
              f"Final Value: ${self.broker.getvalue():,.2f}")

# Compare different trailing stop levels
for trail in [0.03, 0.05, 0.08, 0.10]:
    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=price_data))
    cerebro.addstrategy(TrailingStopStrategy, trail_pct=trail)
    cerebro.broker.setcash(100000)
    cerebro.broker.setcommission(commission=0.001)
    cerebro.run()

### 1.3 Position Sizing Strategies

Position sizing determines how much of your portfolio to allocate per trade.

**Common approaches:**
1. **Fixed size:** Always trade N shares
2. **Fixed fraction:** Trade a fixed percentage of portfolio value
3. **Volatility-based (risk parity):** Size inversely proportional to volatility
4. **Kelly criterion:** Optimal fraction based on win rate and payoff ratio

$$f^* = \frac{p}{a} - \frac{q}{b}$$

where $p$ = probability of winning, $q = 1-p$, $b$ = average win, $a$ = average loss.

In [ ]:
class ATRSizer(bt.Sizer):
    """
    ATR-based position sizer.
    Size = (Capital * Risk%) / (ATR * Multiplier)
    """
    params = dict(
        risk_pct=0.02,
        atr_period=14,
        atr_mult=2.0,
    )
    
    def _getsizing(self, comminfo, cash, data, isbuy):
        if not hasattr(self, '_atr_val'):
            atr = bt.indicators.ATR(data, period=self.p.atr_period)
            self._atr_val = atr
        
        if len(data) < self.p.atr_period + 1:
            return 0
        
        # Calculate ATR manually
        closes = [data.close[-i] for i in range(self.p.atr_period)]
        highs = [data.high[-i] for i in range(self.p.atr_period)]
        lows = [data.low[-i] for i in range(self.p.atr_period)]
        
        trs = []
        for i in range(len(closes)-1):
            tr = max(highs[i]-lows[i], abs(highs[i]-closes[i+1]), abs(lows[i]-closes[i+1]))
            trs.append(tr)
        
        if not trs:
            return 0
        
        atr_val = np.mean(trs)
        stop_distance = atr_val * self.p.atr_mult
        
        if stop_distance <= 0:
            return 0
        
        portfolio_value = self.broker.getvalue()
        risk_amount = portfolio_value * self.p.risk_pct
        size = int(risk_amount / stop_distance)
        max_size = int(cash * 0.95 / data.close[0])
        
        return min(size, max_size)

print("ATRSizer defined - sizes positions based on:")
print("  Size = (Portfolio * Risk%) / (ATR * Multiplier)")
print("  Default: Risk 2% with stop at 2x ATR")

## 2. Parameter Optimization

Backtrader supports parameter optimization through `cerebro.optstrategy()`. This tests all combinations of parameter values and returns results for each.

### 2.1 Grid Search Optimization

In [ ]:
class OptimizableStrategy(bt.Strategy):
    """Strategy with optimizable parameters."""
    params = dict(
        fast_period=20,
        slow_period=50,
    )
    
    def __init__(self):
        self.sma_fast = bt.ind.SMA(self.data.close, period=self.p.fast_period)
        self.sma_slow = bt.ind.SMA(self.data.close, period=self.p.slow_period)
        self.crossover = bt.ind.CrossOver(self.sma_fast, self.sma_slow)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()
    def stop(self):
        print(f"  fast={self.p.fast_period:3d}, slow={self.p.slow_period:3d} => "
              f"Final Value: ${self.broker.getvalue():,.2f}")



# Optimization
cerebro = bt.Cerebro(optreturn=False)
cerebro.adddata(bt.feeds.PandasData(dataname=price_data))

# Define parameter ranges
cerebro.optstrategy(
    OptimizableStrategy,
    fast_period=range(10, 50, 10),     
    slow_period=range(60, 100, 10),   
)

cerebro.broker.setcash(100000)
cerebro.broker.setcommission(commission=0.001)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')

print("Running optimization... (this may take a minute)")
opt_results = cerebro.run(maxcpus=1)
print(f"Tested {len(opt_results)} parameter combinations")

In [ ]:
# Collect results
results_list = []
for run in opt_results:
    strat = run[0]
    sharpe = strat.analyzers.sharpe.get_analysis()
    returns = strat.analyzers.returns.get_analysis()
    
    results_list.append({
        'fast': strat.p.fast_period,
        'slow': strat.p.slow_period,
        'return': returns.get('rtot', 0),
        'sharpe': sharpe.get('sharperatio', 0) or 0,
    })

opt_df = pd.DataFrame(results_list)
opt_df = opt_df.sort_values('sharpe', ascending=False)
print("Top 10 Parameter Combinations by Sharpe Ratio:")
print(opt_df.head(10).to_string(index=False))

### 2.2 Visualizing the Optimization Surface

In [ ]:
# Create heatmap
pivot = opt_df.pivot_table(values='sharpe', index='fast', columns='slow')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sharpe Ratio heatmap
im1 = axes[0].imshow(pivot.values, cmap='RdYlGn', aspect='auto', origin='lower')
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels(pivot.columns.astype(int), rotation=45, fontsize=8)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index.astype(int), fontsize=8)
axes[0].set_xlabel('Slow Period')
axes[0].set_ylabel('Fast Period')
axes[0].set_title('Sharpe Ratio')
fig.colorbar(im1, ax=axes[0])

# Return heatmap
pivot_ret = opt_df.pivot_table(values='return', index='fast', columns='slow')
im2 = axes[1].imshow(pivot_ret.values, cmap='RdYlGn', aspect='auto', origin='lower')
axes[1].set_xticks(range(len(pivot_ret.columns)))
axes[1].set_xticklabels(pivot_ret.columns.astype(int), rotation=45, fontsize=8)
axes[1].set_yticks(range(len(pivot_ret.index)))
axes[1].set_yticklabels(pivot_ret.index.astype(int), fontsize=8)
axes[1].set_xlabel('Slow Period')
axes[1].set_ylabel('Fast Period')
axes[1].set_title('Total Return')
fig.colorbar(im2, ax=axes[1])

plt.suptitle('SMA Crossover Optimization on SPY (2015-2024)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nBest parameters: fast={opt_df.iloc[0]['fast']:.0f}, "
      f"slow={opt_df.iloc[0]['slow']:.0f}, "
      f"Sharpe={opt_df.iloc[0]['sharpe']:.4f}")

## 3. Walk-Forward Analysis

Optimizing on the full dataset leads to **overfitting** — the parameters are fit to historical noise and may not work in the future. Walk-Forward Analysis (WFA) addresses this by:

1. **In-sample (IS):** Optimize parameters on a training window
2. **Out-of-sample (OOS):** Test the best parameters on the next unseen window
3. **Roll forward:** Move the window and repeat

### Types:
- **Rolling WFA:** Fixed-size training window that slides forward
- **Anchored WFA:** Training window starts at the beginning and grows

### Key Principle:
> The only returns that matter are **out-of-sample** returns.

In [ ]:
def walk_forward_analysis(data_df, strategy_class, param_grid,
                          is_window=1000, oos_window=300, anchored=False):
    """
    Perform walk-forward analysis.
    
    Parameters:
    -----------
    data_df : pd.DataFrame - OHLCV data
    strategy_class : bt.Strategy - Strategy class to optimize
    param_grid : dict - Parameter ranges for optimization
    is_window : int - In-sample window size (trading days)
    oos_window : int - Out-of-sample window size
    anchored : bool - If True, in-sample starts from beginning
    
    Returns:
    --------
    pd.DataFrame with WFA results
    """
    total_bars = len(data_df)
    results = []
    step = 0
    
    start_idx = 0
    
    while start_idx + is_window + oos_window <= total_bars:
        step += 1
        
        # Define windows
        if anchored:
            is_start = 0
        else:
            is_start = start_idx
        
        is_end = start_idx + is_window
        oos_start = is_end
        oos_end = min(oos_start + oos_window, total_bars)
        
        is_data = data_df.iloc[is_start:is_end]
        oos_data = data_df.iloc[oos_start:oos_end]
        
        print(f"Step {step}: IS={is_data.index[0].date()} to {is_data.index[-1].date()}, "
              f"OOS={oos_data.index[0].date()} to {oos_data.index[-1].date()}")
        
        # --- Optimize on IS ---
        best_sharpe = -np.inf
        best_params = {}
        
        for fast in param_grid.get('fast_period', [20]):
            for slow in param_grid.get('slow_period', [50]):
                if fast >= slow:
                    continue
                
                cerebro = bt.Cerebro()
                cerebro.adddata(bt.feeds.PandasData(dataname=is_data))
                cerebro.addstrategy(strategy_class, fast_period=fast, slow_period=slow)
                cerebro.broker.setcash(100000)
                cerebro.broker.setcommission(commission=0.001)
                cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
                cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
                
                res = cerebro.run()
                sr = res[0].analyzers.sharpe.get_analysis().get('sharperatio', 0) or 0
                
                if sr > best_sharpe:
                    best_sharpe = sr
                    best_params = {'fast_period': fast, 'slow_period': slow}
        
        # --- Test on OOS with best parameters ---
        cerebro = bt.Cerebro()
        cerebro.adddata(bt.feeds.PandasData(dataname=oos_data))
        cerebro.addstrategy(strategy_class, **best_params)
        cerebro.broker.setcash(100000)
        cerebro.broker.setcommission(commission=0.001)
        cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
        cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
        cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
        
        oos_res = cerebro.run()
        oos_ret = oos_res[0].analyzers.returns.get_analysis()
        oos_sr = oos_res[0].analyzers.sharpe.get_analysis().get('sharperatio', 0) or 0
        
        results.append({
            'step': step,
            'is_start': is_data.index[0].date(),
            'is_end': is_data.index[-1].date(),
            'oos_start': oos_data.index[0].date(),
            'oos_end': oos_data.index[-1].date(),
            'best_fast': best_params['fast_period'],
            'best_slow': best_params['slow_period'],
            'is_sharpe': best_sharpe,
            'oos_return': oos_ret.get('rtot', 0),
            'oos_sharpe': oos_sr,
        })
        
        start_idx += oos_window
    
    return pd.DataFrame(results)

# Run WFA
param_grid = {
    'fast_period': [10, 20, 30, 40],
    'slow_period': [60, 80, 100, 120],
}

print("=" * 70)
print("WALK-FORWARD ANALYSIS (Rolling)")
print("=" * 70)
wfa_results = walk_forward_analysis(
    price_data, OptimizableStrategy, param_grid,
    is_window=1000, oos_window=300, anchored=False
)

In [ ]:
# Analyze WFA results
print("\n" + "=" * 70)
print("WFA RESULTS")
print("=" * 70)
print(wfa_results.to_string(index=False))

# Compound OOS returns
cumulative_oos = (1 + wfa_results['oos_return']).prod() - 1
avg_oos_sharpe = wfa_results['oos_sharpe'].mean()

print(f"\nCompounded OOS Return: {cumulative_oos:.2%}")
print(f"Average OOS Sharpe:    {avg_oos_sharpe:.4f}")

# Plot IS vs OOS Sharpe
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(wfa_results)), wfa_results['is_sharpe'], alpha=0.7, label='In-Sample')
axes[0].bar(range(len(wfa_results)), wfa_results['oos_sharpe'], alpha=0.7, label='Out-of-Sample')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].set_title('IS vs OOS Sharpe Ratio')
axes[0].legend()
axes[0].axhline(y=0, color='k', linestyle='--', linewidth=0.5)

# OOS cumulative return
cum_ret = (1 + wfa_results['oos_return']).cumprod()
axes[1].plot(range(len(cum_ret)), cum_ret, 'b-o')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Walk-Forward OOS Cumulative Return')
axes[1].axhline(y=1, color='k', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Biases in Strategy Development

Awareness of biases is critical to avoid fooling ourselves. The most common biases in quantitative strategy development are:

### 4.1 Look-Ahead Bias
Using information that was not available at the time of the trading decision. Examples: using adjusted data before the adjustment date, using end-of-day data for intraday decisions.

**Prevention:** Backtrader's event-driven architecture prevents most look-ahead bias by design — orders placed in `next()` execute on the **next bar**.

### 4.2 Survivorship Bias
Only using data from securities that currently exist, ignoring those that were delisted, merged, or went bankrupt. This overstates historical returns because the worst performers are excluded.

**Prevention:** Use point-in-time data that includes delisted securities.

### 4.3 Data Mining Bias (Data Snooping)
Testing many parameter combinations and only reporting the best one. With enough trials, you will find patterns in noise.

**Prevention:** Walk-forward analysis, out-of-sample testing, multiple testing correction (Bonferroni, False Discovery Rate).

### 4.4 Overfitting
A model that fits the training data too well, capturing noise rather than signal. Signs: excellent in-sample performance but poor out-of-sample.

**Prevention:** 
- Keep strategies simple (few parameters)
- Use walk-forward analysis
- Test on multiple markets/periods
- Robustness testing

### 4.5 Transaction Cost Bias
Ignoring or underestimating trading costs, especially for high-frequency strategies.

**Prevention:** Always include realistic commission and slippage in backtests.

## 5. Strategy Design Methodology

A systematic approach to building trading strategies:

### Step 1: Hypothesis
Start with an economic or behavioral hypothesis about *why* a strategy should work. A strategy without a logical reason is likely curve-fitted.

### Step 2: Data Collection
Obtain clean data, understand its limitations (survivorship, splits, dividends).

### Step 3: Strategy Specification
Define:
- Universe (which assets to trade)
- Indicators and signals
- Entry and exit rules
- Position sizing
- Risk management (stops, limits)

### Step 4: Backtesting
Implement and test on historical data. Use realistic assumptions for costs.

### Step 5: Optimization
Tune parameters, but be aware of overfitting. Use walk-forward analysis.

### Step 6: Validation
- Out-of-sample testing
- Parameter sensitivity analysis
- Monte Carlo simulation
- Statistical significance tests

### Step 7: Paper Trading
Test in real-time without real money.

### Step 8: Live Trading
Deploy with proper risk controls.

## 6. Exercises

1. **Bracket Orders**: Implement an RSI strategy with bracket orders: buy when RSI < 30, with 2% stop loss and 4% take profit. Test on SPY from 2018-2024.

2. **Trailing Stop Comparison**: Implement the SMA(20/50) crossover on AAPL with trailing stops of 3%, 5%, 8%, and 10%. Which trailing stop level gives the best risk-adjusted return?

3. **Optimization**: Optimize the RSI strategy parameters (RSI period: 7-21, lower threshold: 20-35, upper threshold: 65-80) on SPY. Visualize the optimization surface.

4. **Walk-Forward Analysis**: Perform anchored WFA on the SMA crossover strategy using SPY. Compare the results with rolling WFA. Which approach is more robust?

5. **Bias Identification**: A researcher reports: "I tested 500 parameter combinations on 10 years of SPY data and found one with a Sharpe ratio of 3.5." What biases might be present? What additional tests would you recommend?

---
### References
- Van Tharp, *Trade Your Way to Financial Freedom*
- Robert Pardo, *The Evaluation and Optimization of Trading Strategies*
- [Backtrader Bracket Orders](https://www.backtrader.com/docu/order-creation-execution/bracket/bracket/)
- David Aronson, *Evidence-Based Technical Analysis*